# asrbench on Google Colab (training + evaluation)

Runs `asrbench train` / `run` / `score` on a Colab GPU so the local GPU stays free. Same code, configs and data
as the local repo, so results are comparable (the dataset sha256 in every manifest proves it is the same data).

**Persistence:** checkpoints and run outputs live on Google Drive, so a disconnect loses at most the steps since the
last checkpoint (every 500 steps). Re-run the notebook top to bottom and the train cell resumes automatically.

**One-time setup on the local machine** (packs the already-imported, Russian-filtered data; ~7 GB, FLAC does not compress):
```bash
tar -cf asr-data.tar data/ksc2 data/fleurs/kk_kz data/fleurs/ka_ge
```
Upload `asr-data.tar` to `MyDrive/asr-project3/` (Drive for desktop or the web UI).

**GPU notes:** T4 (free tier) has no bf16; `asrbench train` switches to fp16 automatically. L4/A100 keep bf16.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings

In [ ]:
REPO_URL   = "https://github.com/Efeblk/asr-project3.git"
BRANCH     = "main"                                            # branch to train from
DRIVE_DIR  = "/content/drive/MyDrive/asr-project3"             # holds asr-data.tar, checkpoints/, runs/
CONFIG     = "configs/train/kk/whisper-small-ft_ksc2-sample.yaml"
EVAL_CONFIGS = []   # optional run configs to evaluate after training, e.g. ["configs/runs/kk/whisper-small-ft_ksc2-dev.yaml"]

## 2. Drive, code, packages

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
pathlib.Path(DRIVE_DIR, "checkpoints").mkdir(parents=True, exist_ok=True)
pathlib.Path(DRIVE_DIR, "runs").mkdir(parents=True, exist_ok=True)

In [ ]:
# Private repo: add a GitHub token as a Colab secret named GITHUB_TOKEN (key icon in the left bar). Public: nothing to do.
import subprocess
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("/content/asr-project3"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/content/asr-project3"], check=True)
%cd /content/asr-project3
!git log --oneline -1

In [ ]:
# Keep Colab's preinstalled CUDA torch; pin the libraries the local runs were verified with.
!pip -q install -e ".[hf]" "transformers==4.57.6" "accelerate==1.12.0" pytest
!python -m pytest -q 2>&1 | tail -1

## 3. Data (copied to the fast local disk) and Drive-backed outputs

In [ ]:
if not os.path.exists("data/ksc2/manifest.jsonl"):
    !tar -xf "{DRIVE_DIR}/asr-data.tar" -C /content/asr-project3
!wc -l data/ksc2/manifest.jsonl
# checkpoints/ and runs/ point at Drive so they survive a disconnect
for d in ("checkpoints",):
    if not os.path.islink(d):
        !rm -rf {d} && ln -s "{DRIVE_DIR}/{d}" {d}
!mkdir -p "{DRIVE_DIR}/runs" && cp -rn runs/. "{DRIVE_DIR}/runs/" 2>/dev/null; rm -rf runs && ln -s "{DRIVE_DIR}/runs" runs
!ls -la checkpoints runs | head

## 4. Train (resumes the latest unfinished run of this config automatically)

In [ ]:
import yaml, glob
name = yaml.safe_load(open(CONFIG, encoding="utf-8"))["name"]
unfinished = sorted(d for d in glob.glob(f"checkpoints/{name}_*")
                    if glob.glob(f"{d}/checkpoint-*") and not os.path.exists(f"{d}/final"))
resume = f"--resume {unfinished[-1]}" if unfinished else ""
print("resume:", resume or "(fresh run)")
!asrbench train {CONFIG} {resume}

## 5. Evaluate the saved model (optional) and score

In [ ]:
for cfg in EVAL_CONFIGS:
    !asrbench run {cfg}
latest = sorted(glob.glob("runs/*_20*"), key=os.path.getmtime)
for r in latest[-len(EVAL_CONFIGS):] if EVAL_CONFIGS else []:
    !asrbench score {r}

## 6. Bring results home
Everything is already on Drive (`MyDrive/asr-project3/runs/` and `checkpoints/`). On the local machine, copy the new
`runs/<name>_<ts>/` and `runs/train_<name>_<ts>/` folders into the repo and commit them; copy
`checkpoints/<name>_<ts>/final` into the local `checkpoints/` to run the model locally. (Weights stay out of git.)